# Vision-Language-Action notebook

This notebook extends the Ollama chatbot pattern with RGBD perception and a guarded action model. It detects a coffee mug on the table, visualizes the mug and end-effector segmentation, asks an Ollama vision model for a structured grasp target, predicts the next `ik_ee_pose`, and only moves the arm when Execute is pressed.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import Ollama, RGBD, notebook display, and IK helpers.


In [ ]:
import base64
import json
import math
import re
import time
import urllib.error
import urllib.request

import cv2
import ipywidgets as widgets
import numpy as np
from IPython.display import display

from arm_sdk import ArmSdk
from sdk_client import Robot


Configure the local models and robot clients. The vision model must support Ollama image input.


In [ ]:
OLLAMA_URL = os.environ.get("G1_OLLAMA_URL", os.environ.get("OLLAMA_HOST", "http://192.168.123.164:11434")).rstrip("/")
CHAT_MODEL = os.environ.get("G1_CHAT_MODEL", "granite4.1:3b")
VISION_MODEL = os.environ.get("G1_VISION_MODEL", "qwen3-vl:8b")
RGBD_HOST = os.environ.get("G1_RGBD_HOST", "192.168.2.41")
RGBD_PORT = int(os.environ.get("G1_RGBD_PORT", "5555"))
RGBD_TOPIC = os.environ.get("G1_RGBD_TOPIC", "")
ARM = os.environ.get("G1_VLA_ARM", "right")

robot = Robot(
    iface=IFACE,
    domain_id=DOMAIN_ID,
    safety_boot=False,
    recover_dev_mode_on_init=False,
    auto_start_sensors=False,
    rgbd_host=RGBD_HOST,
    rgbd_port=RGBD_PORT,
    rgbd_topic=RGBD_TOPIC,
)
ik = ArmSdk(iface=IFACE, domain_id=DOMAIN_ID)
ik.resync()

vla_messages = [{
    "role": "system",
    "content": "You are a concise robot manipulation assistant. Return compact, valid JSON when asked.",
}]
last_prediction = None
print(f"VLA ready: ollama={OLLAMA_URL} chat={CHAT_MODEL} vision={VISION_MODEL} rgbd=tcp://{RGBD_HOST}:{RGBD_PORT}")


Ollama helpers copied from the chatbot notebook, plus JSON extraction for model outputs.


In [ ]:
def clean_reply(text):
    # TODO: Strip hidden thinking blocks and normalize whitespace before displaying or speaking the model reply.
    raise NotImplementedError("Participant exercise: complete clean_reply.")


def post_ollama_chat(body, timeout=45.0):
    # TODO: POST JSON to Ollama /api/chat and handle HTTP errors with useful messages.
    raise NotImplementedError("Participant exercise: complete post_ollama_chat.")


def extract_json_object(text):
    # TODO: Find a JSON object in plain or fenced model text and parse it.
    raise NotImplementedError("Participant exercise: complete extract_json_object.")


Perception helpers. The classical detector finds a colored object blob and estimates depth; the vision model can refine the semantic label and target choice.


In [ ]:
def jpeg_data_url(bgr):
    # TODO: JPEG-encode the OpenCV BGR image and wrap it as a browser data URL.
    raise NotImplementedError("Participant exercise: complete jpeg_data_url.")


def mask_data_url(mask):
    # TODO: Convert a grayscale mask into a browser-displayable JPEG data URL.
    raise NotImplementedError("Participant exercise: complete mask_data_url.")


def colorize_depth(depth_m, max_depth_m=4.0):
    # TODO: Convert valid depth values into an 8-bit display image and apply a colormap.
    raise NotImplementedError("Participant exercise: complete colorize_depth.")


def segment_hsv_depth(rgb_bgr, depth_m, hsv_low, hsv_high, *, min_area_px=300, max_depth_m=2.5, roi=None, label="segment", color=(0, 255, 255)):
    # TODO: Threshold HSV color, combine it with valid depth, clean the mask, and extract contour detections.
    raise NotImplementedError("Participant exercise: complete segment_hsv_depth.")


def detect_table_object(rgb_bgr, depth_m, hsv_low, hsv_high, min_area_px=500, max_depth_m=2.0):
    # TODO: Use HSV/depth segmentation and return the best object detection plus overlay.
    raise NotImplementedError("Participant exercise: complete detect_table_object.")


def detect_end_effector(rgb_bgr, depth_m, hsv_low, hsv_high, min_area_px=300, max_depth_m=2.5):
    # TODO: Limit segmentation to the likely hand area and return the best end-effector detection.
    raise NotImplementedError("Participant exercise: complete detect_end_effector.")


def perception_overlay(rgb_bgr, depth_m, object_hsv_low, object_hsv_high, ee_hsv_low, ee_hsv_high, min_area_px=500, max_depth_m=2.0):
    # TODO: Combine object and end-effector detections into masks, overlays, and a context dict.
    raise NotImplementedError("Participant exercise: complete perception_overlay.")


def rotation_matrix_to_rpy(R):
    # TODO: Convert a 3x3 rotation matrix into roll, pitch, yaw with singularity handling.
    raise NotImplementedError("Participant exercise: complete rotation_matrix_to_rpy.")


def current_pose(arm=ARM):
    # TODO: Read the selected arm transform and return xyz+rpy in base_link coordinates.
    raise NotImplementedError("Participant exercise: complete current_pose.")


VLA prediction. The model sees the RGB image plus detector/depth context and must return the next target pose as JSON.


In [ ]:
VLA_SYSTEM = """You are controlling a Unitree humanoid arm in base_link coordinates.
Return only valid JSON. Do not include markdown.
Predict a cautious next ik_ee_pose to position the selected hand around the detected coffee mug on the table.
The pose is [x, y, z, roll, pitch, yaw] in meters/radians. Keep one step small and reachable.
Prefer a pre-grasp pose 8 to 12 cm from the object, slightly above its center, with a neutral wrist.
If detection is unsafe or absent, set safe_to_move false and keep the current pose."""

VLA_SCHEMA = {
    "object_detected": True,
    "object_label": "coffee mug",
    "object_center_px": [0, 0],
    "confidence": 0.0,
    "next_ik_ee_pose": [0, 0, 0, 0, 0, 0],
    "safe_to_move": False,
    "reason": "short explanation",
}


def ask_vision_action(rgb_jpeg, context):
    # TODO: Send the RGB frame and context to the vision model, then parse the JSON action.
    raise NotImplementedError("Participant exercise: complete ask_vision_action.")


def sanitize_prediction(prediction, fallback_pose):
    # TODO: Validate the model pose, clamp each step, and mark unsafe predictions as not executable.
    raise NotImplementedError("Participant exercise: complete sanitize_prediction.")


def pose_increment_from_prediction(prediction, current_xyzrpy):
    # TODO: Subtract the current pose from the predicted target pose.
    raise NotImplementedError("Participant exercise: complete pose_increment_from_prediction.")



def ramped_ik_move_EE(
    pose_increment,
    *,
    arm=ARM,
    max_dq=0.04,
    cart_speed_m_s=0.04,
    rot_speed_rad_s=0.25,
    rate_hz=20.0,
):
    # TODO: Break a Cartesian pose increment into safe IK steps and send them at a fixed rate.
    raise NotImplementedError("Participant exercise: complete ramped_ik_move_EE.")


Run the panel. Tune HSV for the table object, then Detect and Predict. Execute sends a single guarded IK increment.


In [ ]:
h_low = widgets.IntSlider(value=0, min=0, max=179, description="H low")
h_high = widgets.IntSlider(value=179, min=0, max=179, description="H high")
s_low = widgets.IntSlider(value=0, min=0, max=255, description="S low")
s_high = widgets.IntSlider(value=255, min=0, max=255, description="S high")
v_low = widgets.IntSlider(value=80, min=0, max=255, description="V low")
v_high = widgets.IntSlider(value=255, min=0, max=255, description="V high")
ee_h_low = widgets.IntSlider(value=0, min=0, max=179, description="EE H low")
ee_h_high = widgets.IntSlider(value=179, min=0, max=179, description="EE H high")
ee_s_low = widgets.IntSlider(value=0, min=0, max=255, description="EE S low")
ee_s_high = widgets.IntSlider(value=255, min=0, max=255, description="EE S high")
ee_v_low = widgets.IntSlider(value=0, min=0, max=255, description="EE V low")
ee_v_high = widgets.IntSlider(value=90, min=0, max=255, description="EE V high")
min_area = widgets.IntSlider(value=500, min=50, max=10000, step=50, description="Area")
max_depth = widgets.FloatSlider(value=2.0, min=0.2, max=6.0, step=0.1, description="Depth m")
max_dq = widgets.FloatSlider(value=0.04, min=0.01, max=0.12, step=0.005, description="max dq")
cart_speed = widgets.FloatSlider(value=0.04, min=0.005, max=0.12, step=0.005, description="m/s")
rot_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05, description="rad/s")
ramp_rate = widgets.FloatSlider(value=20.0, min=5.0, max=50.0, step=1.0, description="Hz")
detect = widgets.Button(description="Detect and Predict", button_style="success")
execute = widgets.Button(description="Execute IK Step", button_style="warning")
resync = widgets.Button(description="Resync IK")
perception_img = widgets.HTML(value="")
depth_img = widgets.HTML(value="")
object_mask_img = widgets.HTML(value="")
ee_mask_img = widgets.HTML(value="")
status = widgets.Textarea(layout=widgets.Layout(width="100%", height="340px"), disabled=True)


def set_status(payload):
    # TODO: Serialize status payloads into the notebook text area.
    raise NotImplementedError("Participant exercise: complete set_status.")


def on_detect(_=None):
    # TODO: Capture a frame, run perception, ask the VLA model, and store/display the prediction.
    raise NotImplementedError("Participant exercise: complete on_detect.")


def on_execute(_):
    # TODO: Validate the stored prediction, compute the pose increment, and run the IK step.
    raise NotImplementedError("Participant exercise: complete on_execute.")


def on_resync(_):
    # TODO: Resynchronize IK state and refresh the status view.
    raise NotImplementedError("Participant exercise: complete on_resync.")

detect.on_click(on_detect)
execute.on_click(on_execute)
resync.on_click(on_resync)
set_status({"status": "ready", "arm": ARM})
display(widgets.VBox([
    widgets.HBox([h_low, h_high, s_low, s_high, v_low, v_high]),
    widgets.HBox([ee_h_low, ee_h_high, ee_s_low, ee_s_high, ee_v_low, ee_v_high]),
    widgets.HBox([min_area, max_depth, max_dq]),
    widgets.HBox([cart_speed, rot_speed, ramp_rate, detect, execute, resync]),
    status,
    widgets.HBox([perception_img, depth_img]),
    widgets.HBox([object_mask_img, ee_mask_img]),
]))
